In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance


import json, os, joblib


RANDOM_STATE = 42


os.makedirs("artifacts/figures", exist_ok=True)

In [ ]:
df = pd.read_csv("S06-hw-dataset-04.csv")


print(df.head())
print(df.info())
print(df.describe())


print("\nРаспределение target:")
print(df["target"].value_counts(normalize=True))


X = df.drop(columns=["target", "id"])
y = df["target"]

   id       f01       f02       f03       f04       f05       f06       f07  \
0   1 -1.250210  1.423474 -0.225004 -4.023138 -0.832729 -0.550874  1.772090   
1   2  0.074328  0.376429  0.212831 -0.502074  2.017405  0.625496  1.943785   
2   3  0.638481  0.060968  0.746760  2.479653 -0.292858 -0.078139 -2.918423   
3   4  1.712916 -1.350969 -0.256473  1.622074 -0.445141  0.911932 -3.440345   
4   5  0.905676 -0.206545 -0.068806  4.086026 -1.010045 -0.772644 -4.207688   

        f08       f09  ...        f52        f53       f54       f55  \
0  2.761690 -0.698750  ...  10.938269   0.501178  1.600001  0.314212   
1  1.242030 -0.524090  ...   7.775262  -4.550195  6.272586 -0.932162   
2 -0.013186  1.009135  ...  -4.448447  -9.593179 -3.093519  0.029321   
3  1.505192 -1.104348  ...  -1.619072  -3.237479 -5.474038 -1.582475   
4  2.506104  1.589143  ...  -2.396844 -10.540129 -5.532811 -1.231203   

        f56       f57       f58       f59       f60  target  
0  1.209735  1.355697 -5.33892

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)


print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [3]:
models = {}
results = {}


models["Dummy"] = DummyClassifier(strategy="most_frequent")
models["LogReg"] = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE))
])


for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:,1]
    pred = model.predict(X_test)
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba)
    }

In [4]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced")


dt_grid = {
    "max_depth": [3, 5, 8, None],
    "min_samples_leaf": [1, 5, 20],
}


dt_search = GridSearchCV(dt, dt_grid, scoring="roc_auc", cv=5, n_jobs=-1)
dt_search.fit(X_train, y_train)


best_dt = dt_search.best_estimator_

In [5]:
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
rf_grid = {
    "n_estimators": [200],
    "max_depth": [None, 8],
    "min_samples_leaf": [1, 10],
    "max_features": ["sqrt", 0.5]
}


rf_search = GridSearchCV(rf, rf_grid, scoring="roc_auc", cv=5, n_jobs=-1)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

In [7]:
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
gb_grid = {
    "n_estimators": [200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3]
}


gb_search = GridSearchCV(gb, gb_grid, scoring="roc_auc", cv=5, n_jobs=18)
gb_search.fit(X_train, y_train)
best_gb = gb_search.best_estimator_

In [8]:
final_models = {
"DecisionTree": best_dt,
"RandomForest": best_rf,
"GradientBoosting": best_gb
}


for name, model in final_models.items():
    proba = model.predict_proba(X_test)[:,1]
    pred = model.predict(X_test)
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba)
    }


with open("artifacts/metrics_test.json", "w") as f:
    json.dump(results, f, indent=2)


search_summaries = {
    "DecisionTree": {"best_params": dt_search.best_params_, "cv_score": dt_search.best_score_},
    "RandomForest": {"best_params": rf_search.best_params_, "cv_score": rf_search.best_score_},
    "GradientBoosting": {"best_params": gb_search.best_params_, "cv_score": gb_search.best_score_}
}


with open("artifacts/search_summaries.json", "w") as f:
    json.dump(search_summaries, f, indent=2)

In [9]:
best_name = max(results, key=lambda k: results[k]["roc_auc"])
best_model = final_models.get(best_name, models.get(best_name))


joblib.dump(best_model, "artifacts/best_model.joblib")


with open("artifacts/best_model_meta.json", "w") as f:
    json.dump({"best_model": best_name, "metrics": results[best_name]}, f, indent=2)

In [10]:
proba = best_model.predict_proba(X_test)[:,1]
RocCurveDisplay.from_predictions(y_test, proba)
plt.savefig("artifacts/figures/roc_curve.png", dpi=150)
plt.close()


cm = confusion_matrix(y_test, best_model.predict(X_test))
ConfusionMatrixDisplay(cm).plot()
plt.savefig("artifacts/figures/confusion_matrix.png", dpi=150)
plt.close()

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(15)
print("Top-15 признаков по permutation importance:")
print(imp)


plt.figure(figsize=(8,6))
imp[::-1].plot(kind="barh")
plt.title("Permutation importance (Top-15)")
plt.tight_layout()
plt.savefig("artifacts/figures/permutation_importance.png", dpi=150)
plt.close()


print("Готово! Все артефакты сохранены в папке artifacts/.")

Top-15 признаков по permutation importance:
f58    0.010496
f53    0.008960
f13    0.007872
f47    0.007136
f25    0.005312
f54    0.005216
f38    0.004320
f11    0.004096
f41    0.003680
f57    0.003072
f52    0.002464
f33    0.002272
f08    0.002176
f04    0.002112
f43    0.001952
dtype: float64

Готово! Все артефакты сохранены в папке artifacts/.
